In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
X_train=train[features_x]
y_train=train[features_y]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_train))

models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    weights = np.where(
        y_tr > y_tr.quantile(0.95),
        2,
        1
    )
    
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        random_state=42 + fold
    )
    
    model.fit(
        X_tr,
        np.log1p(y_tr),
        sample_weight=weights
    )
    
    y_pred = np.expm1(model.predict(X_val))
    oof_preds[val_idx] = y_pred
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"Fold {fold} RMSE: {rmse}")
    
    # 👉 STORE MODEL
    models.append(model)